In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

predictions = []
y_test_list = []
dates = []
dic_r2_all = {}
num_coef_time = {}

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_val = val_data['RET'].values
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values

    alphas = np.linspace(start=0.00001, stop=0.004, num=20)
    mse = np.full((len(alphas), 1), np.nan, dtype=np.float32)

    for i, alpha in enumerate(alphas):
        model_val = ElasticNet(alpha=alpha, l1_ratio=0.5)
        model_val.fit(X_train, y_train)
        Yval_predict = model_val.predict(X_val)
        mse[i, 0] = mean_squared_error(y_val, Yval_predict)

    optim_alpha = alphas[np.argmin(mse)]
    model = ElasticNet(alpha=optim_alpha, l1_ratio=0.5)
    model.fit(np.concatenate((X_train, X_val)), np.concatenate((y_train, y_val)))
    preds = model.predict(X_test)

    predictions.append(preds)
    dates.append(test_data['DATE'].values)
    y_test_list.append(y_test)
    r2 = 1 - np.sum((y_test - preds) ** 2) / np.sum(y_test ** 2)
    dic_r2_all[f"r2.{year}"] = r2

    num_coef = np.count_nonzero(model.coef_)
    num_coef_time[f"ncoef.{year}"] = num_coef

predictions_all = np.concatenate(predictions, axis=0)
y_test_list_all = np.concatenate(y_test_list, axis=0)
dates_all = np.concatenate(dates, axis=0)

R2OOS_ENet = 1 - np.sum((y_test_list_all - predictions_all) ** 2) / np.sum(y_test_list_all ** 2)
print("R2OOS Elastic net: ", R2OOS_ENet)

df_intermediate = pd.DataFrame({
    'Date': dates_all,
    'Prediction': predictions_all,
    'True Value': y_test_list_all
})

df_intermediate = df_intermediate.assign(**dic_r2_all)
df_intermediate = df_intermediate.assign(**num_coef_time)

print(df_intermediate.head())